In [ ]:
import pybryt

test_arr = [2, 3, 4, 10, 40]
test_x = 10

# --- 1. 產生迭代版 (不包含時間複雜度) ---
def teacher_logic_algo(arr, l, r, x):
    ref_values = []
    while (l <= r):
        m = (l + r) // 2
        ref_values.append(pybryt.Value(m)) 
        if (arr[m] == x): break
        if (arr[m] < x): l = m + 1
        else: r = m - 1
    return ref_values

algo_data = teacher_logic_algo(test_arr, 0, len(test_arr)-1, test_x)
ref_algo = pybryt.ReferenceImplementation("algo", algo_data, display_name="迭代版二元搜尋")
ref_algo.dump("binarysearch_algo.pkl")

# --- 2. 產生遞迴版 (不包含時間複雜度) ---
def teacher_logic_rec(arr, l, r, x, refs):
    if l <= r:
        m = (l + r) // 2
        refs.append(pybryt.Value(m))
        if arr[m] == x: return m
        elif arr[m] > x: return teacher_logic_rec(arr, l, m - 1, x, refs)
        else: return teacher_logic_rec(arr, m + 1, r, x, refs)
    return -1

rec_data = []
teacher_logic_rec(test_arr, 0, len(test_arr)-1, test_x, rec_data)
ref_rec = pybryt.ReferenceImplementation("recursive", rec_data, display_name="遞迴版二元搜尋")
ref_rec.dump("binarysearch_recursive.pkl")

print("✅ 老師的兩把尺已重新打磨完畢（移除效能要求，專注邏輯比對）")

✅ 老師的兩把尺已重新打磨完畢（移除效能要求，專注邏輯比對）


In [18]:
# 模擬學生寫的程式碼內容
student_code_content = """
def student_binary_search(arr, l, r, x):
    while (l <= r):
        m = (l + r) // 2
        if (arr[m] == x):
            return m
        if (arr[m] < x):
            l = m + 1
        else:
            r = m - 1
    return -1
"""

# 把這段內容存成實體檔案 subm.py
with open("subm.py", "w", encoding="utf-8") as f:
    f.write(student_code_content)

print("✅ 學生作業 subm.py 已存檔成功！")

✅ 學生作業 subm.py 已存檔成功！


In [19]:
import pybryt
import pybryt.execution.memory_footprint as mfp

# 1. 載入剛才產生的純淨版尺
ref_algo = pybryt.ReferenceImplementation.load('binarysearch_algo.pkl')
ref_rec = pybryt.ReferenceImplementation.load('binarysearch_recursive.pkl')

# 2. 準備足跡物件
footprint = pybryt.MemoryFootprint()
my_values = []  # 🔥 核心關鍵：我們自己保管資料，不信任 PyBryt 內建的清單

test_arr = [2, 3, 4, 10, 40]
test_x = 10
l_copy, r_copy = 0, len(test_arr) - 1
timestamp = 0

print("===== 🚀 正在啟動終極綁架版比對 =====")

# 模擬迭代版執行
while (l_copy <= r_copy):
    m = (l_copy + r_copy) // 2
    
    val = mfp.MemoryFootprintValue(m, timestamp, "line")
    footprint.add_value(val) # 塞給它（雖然它內部壞了）
    my_values.append(val)    # 🔥 備份到我們自己的清單裡！
    
    timestamp += 1
    if (test_arr[m] == test_x): break
    if (test_arr[m] < test_x): l_copy = m + 1
    else: r_copy = m - 1

# 🔥🔥🔥 【神級覆寫】：強制綁架 footprint 的讀取功能！
# 當 PyBryt 試圖用 for 迴圈讀取足跡時，強制它讀取我們 100% 正常的 my_values 陣列
footprint.__class__.__iter__ = lambda self: iter(my_values)

# 3. 進行比對
res_algo = ref_algo.run(footprint)
res_rec = ref_rec.run(footprint)

# 4. 印出雙重報告
print("\n" + "="*40)
print(f"📊 批改結果 1: {ref_algo.display_name}")
print(f"✅ 是否通過: {'是' if res_algo.correct else '否'}")

print("-" * 40)

print(f"📊 批改結果 2: {ref_rec.display_name}")
print(f"✅ 是否通過: {'是' if res_rec.correct else '否'}")
print("="*40)

===== 🚀 正在啟動終極綁架版比對 =====

📊 批改結果 1: 迭代版二元搜尋
✅ 是否通過: 是
----------------------------------------
📊 批改結果 2: 遞迴版二元搜尋
✅ 是否通過: 是


In [12]:
import numpy as np
import pybryt

def track_fibonacci(n):
    if n <= 0:
        return np.array([])
        
    fibs = np.zeros(n, dtype=int) 

    fibs[0] = 0
    curr_val = pybryt.Value(fibs) 
    
    if n == 1:
        return fibs
        

    fibs[1] = 1
    v = pybryt.Value(fibs)         
    curr_val.before(v)            
    curr_val = v                 
    
    if n == 2:
        return fibs
    print(fibs)
    
    for i in range(2, n):
        fibs[i] = fibs[i-1] + fibs[i-2]

        v = pybryt.Value(fibs)  
        curr_val.before(v)     
        curr_val = v
    print(fibs)

track_fibonacci(5)

[0 1 0 0 0]
[0 1 1 2 3]


In [20]:
import numpy as np
import pybryt

def fff(n_fibs):
    first_ref = []
    second_ref = []

    fibs = np.zeros(n_fibs, dtype=int)

    fibs[0] = 0
    curr_val = pybryt.Value(fibs)    
    first_ref.append(curr_val)       

    if n_fibs == 1:
        return fibs

    fibs[1] = 1
    v = pybryt.Value(fibs)
    first_ref.append(curr_val.before(v)) 
    curr_val = v
    
    if n_fibs == 2:
        return fibs

   
    for i in range(2, n_fibs):   
        fibs[i] = fibs[i-1] + fibs[i-2]

        v = pybryt.Value(fibs)
        first_ref.append(curr_val.before(v)) 
        curr_val = v
        
    return fibs  


result = fff(50)  
print(result)     

[         0          1          1          2          3          5
          8         13         21         34         55         89
        144        233        377        610        987       1597
       2584       4181       6765      10946      17711      28657
      46368      75025     121393     196418     317811     514229
     832040    1346269    2178309    3524578    5702887    9227465
   14930352   24157817   39088169   63245986  102334155  165580141
  267914296  433494437  701408733 1134903170 1836311903 2971215073
 4807526976 7778742049]


In [23]:
second_ref = []
fib_map = {}

def fib(n):
    if n == 0:
        return 0

    if n == 1:
        return 1

    if n in fib_map:
        return fib_map[n]
    ans = fib(n-1) + fib(n-2)
    fib_map[n] = ans 
    return ans   


# 呼叫函式啟動計算
print(fib(10))

55


In [ ]:
import numpy as np
import pybryt

n_fibs = 50

first_ref = []
fibs = np.zeros(n_fibs, dtype=int)


fibs[0] = 0
curr_val = pybryt.Value(fibs)
first_ref.append(curr_val)

if n_fibs > 1:
   
    fibs[1] = 1
    v = pybryt.Value(fibs)
    first_ref.append(curr_val.before(v))
    curr_val = v


for i in range(2, n_fibs):
    fibs[i] = fibs[i-1] + fibs[i-2]
    v = pybryt.Value(fibs)
    first_ref.append(curr_val.before(v))
    curr_val = v

final_answer_array = fibs[-1] 



second_ref = []
fib_map = {}

def fib(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    if n in fib_map:
        return fib_map[n]
        
    ans = fib(n-1) + fib(n-2)
    fib_map[n] = ans
    
   
    second_ref.append(pybryt.Value(fib_map))
    return ans


final_answer_dict = fib(n_fibs)



ref1 = pybryt.ReferenceImplementation("ref1_array", first_ref)
ref2 = pybryt.ReferenceImplementation("ref2_dict", second_ref)



print(len(first_ref))
print(len(second_ref))

50
49


In [5]:
%pip install pybryt

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------- ----------------- 1.6/2.8 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 8.9 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.7 MB 11.2 MB/s eta 0:00:01
   ------------------- -------------------- 4.7/9.7 MB 11.4 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.7 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 11.4 MB/s  0:00:00

   - --------------------------------------  1/27 [fastjsonschema]
   -- -------------------------------------  2/27 [wheel]
   -- -------------------------------------  2/27 [wheel]
   ---- -----------------------------------  3/27 [tzdata]
   ---- --


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
